# Model 3: Random forest

Predict 14 units for each hour, then reshape to the submission format.
Features describe the current price relative to water value and competing hours in the week.
Course references: `Feature_Engineering_slides.pdf` (differences, ratios and window statistics),
and `predictor_practice_related_materials/rf_iris.ipynb`.


## 1. Read the data

In [1]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from kaggle_metric import score


KeyboardInterrupt: 

In [ ]:
PROJECT_ROOT = Path.cwd()
KERNEL = PROJECT_ROOT / 'data' / 'kernel'
EXTENDED = PROJECT_ROOT / 'data' / 'extended'

train = pd.read_csv(KERNEL / 'Unit_commitment_decisions.csv')
train['starttime'] = pd.to_datetime(train['starttime'], utc=True).dt.tz_localize(None)
target_columns = [c for c in train.columns if c.startswith('result_committed_')]

def read_time_table(path):
    table = pd.read_csv(path)
    table['date'] = pd.to_datetime(table['date'], utc=True).dt.tz_localize(None)
    return table.set_index('date').sort_index()

price = read_time_table(KERNEL / 'Historical_day_ahead_price_2015_2025.csv')
inflow = read_time_table(KERNEL / 'Historical_inflow_1958_2025.csv')
volume = read_time_table(KERNEL / 'Historical_volume_2015_2024.csv')
water_value = read_time_table(KERNEL / 'Synthetic_water_value_2015_2024.csv')
min_volume = read_time_table(EXTENDED / 'Constraint_min_volume.csv')
min_flow = read_time_table(EXTENDED / 'Constraint_min_flow.csv')


## 2. Build hourly features

Use end-of-week water values, as specified by the dataset. Summarize weekly
price margins and the fraction of hours priced above each water value.
A 24-hour price average adds context around the current hour.
Omit raw price levels, start water values, and the two constant storage columns.
Domain reference: [SINTEF SHOP water values](https://docs.shop.sintef.energy/examples/water_values/water_values.html).
Also measure nearby profitable hours, price changes, and position within the forecast week.
Weekly inflow, its ratio to starting storage, and storage above minimum requirements
provide additional water-availability context for Haukeli.


In [ ]:
HORIZON = 168

def weekly_window(table, starts):
    timestamps = starts.to_numpy()[:, None] + np.arange(HORIZON, dtype='timedelta64[h]')
    values = table.reindex(pd.DatetimeIndex(timestamps.ravel())).to_numpy()
    return values.reshape(len(starts), HORIZON, table.shape[1])

def make_features(cases):
    starts = cases['starttime']
    prices = weekly_window(price, starts)[:, :, 0]
    mean_price = prices.mean(axis=1, keepdims=True)
    spread = prices.std(axis=1, keepdims=True) + 1
    features = {
        'hour': np.tile(np.arange(HORIZON) % 24, len(cases)),
        'day': np.tile(np.arange(HORIZON) // 24, len(cases)),
        'month': np.repeat(starts.dt.month.to_numpy(), HORIZON),
        'day_of_year': np.repeat(starts.dt.dayofyear.to_numpy(), HORIZON),
        'relative_price': ((prices - mean_price) / spread).ravel(),
        'price_rank': pd.DataFrame(prices).rank(axis=1, pct=True).to_numpy().ravel(),
    }
    storage = volume.drop(columns=['Hyljelihyl', 'Vatjern']).reindex(pd.DatetimeIndex(starts))
    for name in storage:
        features[f'volume_{name}'] = np.repeat(storage[name].to_numpy(), HORIZON)

    end_values = water_value.reindex(pd.DatetimeIndex(starts + pd.Timedelta(days=7)))
    weekly = {}
    nearby = {}
    for name in end_values:
        value = end_values[name].to_numpy()[:, None]
        features[f'margin_water_end_{name}'] = ((prices - value) / spread).ravel()
        weekly[f'weekly_margin_{name}'] = np.repeat(((mean_price - value) / spread).ravel(), HORIZON)
        weekly[f'profitable_share_{name}'] = np.repeat((prices > value).mean(axis=1), HORIZON)
        profitable = pd.DataFrame(prices > value)
        nearby[f'local_profitable_{name}'] = profitable.T.rolling(12, center=True, min_periods=1).mean().T.to_numpy().ravel()

    for prefix, table in [('inflow', inflow), ('min_volume', min_volume), ('min_flow', min_flow)]:
        values = weekly_window(table, starts)
        for j, name in enumerate(table.columns):
            features[f'{prefix}_{name}'] = values[:, :, j].ravel()
    features.update(weekly)
    local_price = pd.DataFrame(prices).T.rolling(24, center=True, min_periods=1).mean().T.to_numpy()
    features['local_price'] = ((local_price - mean_price) / spread).ravel()
    features['price_rise'] = (np.diff(prices, axis=1, prepend=prices[:, :1]) / spread).ravel()
    features['next_price_change'] = (np.diff(prices, axis=1, append=prices[:, -1:]) / spread).ravel()
    features['horizon_hour'] = np.tile(np.arange(HORIZON), len(cases))
    features['relative_spread'] = np.repeat((spread / (np.abs(mean_price) + 1)).ravel(), HORIZON)
    features.update(nearby)
    initial = volume.reindex(pd.DatetimeIndex(starts))
    for name in volume:
        flow = weekly_window(inflow[[name]], starts)[:, :, 0]
        weekly_flow = flow.sum(axis=1) * 3600 / 1e6
        features[f'weekly_inflow_{name}'] = np.repeat(flow.mean(axis=1), HORIZON)
        features[f'inflow_ratio_{name}'] = np.repeat(weekly_flow / (initial[name].to_numpy() + 0.1), HORIZON)
    for name in min_volume:
        minimum = weekly_window(min_volume[[name]], starts)[:, :, 0]
        features[f'headroom_{name}'] = (initial[name].to_numpy()[:, None] - minimum).ravel()
    return pd.DataFrame(features, dtype=np.float32)


## 3. Train and validate

Fit forests separately for each unit. Hogga and Kjela use the full core feature set;
Haukeli also uses water-availability features. Vinje uses deeper trees.
Lio averages forests using local and broader context. Three units train concurrently.
The 2022 validation split has a seven-day gap.

Measured micro ROC-AUC: **2021: 0.9793; 2022: 0.9794**.
Both years informed model development.


In [ ]:
reservoirs = {
    'Hogga': ['Bandak'], 'Tokke': ['Vinjevatn', 'Vaamarvatn', 'Bandak'],
    'Lio': ['Byrtevatn', 'Botnedalsvatn', 'Bandak'], 'Byrte': ['Botnedalsvatn', 'Byrtevatn'],
    'Vinje': ['Vaamarvatn', 'Totak', 'Vinjevatn'], 'Haukeli': ['Vatjern', 'Langeidvatn'],
    'Songa': ['Songavatn', 'Bitdalsvatn', 'Vaamarvatn'],
    'Kjela': ['Foersvatn', 'Bordalsvatn', 'Venemo'],
    'Vesle Kjela': ['Kjelavatn', 'Foersvatn', 'Staavatn'],
}
units = [c.removeprefix('result_committed_').rsplit('_t', 1)[0] for c in target_columns[::HORIZON]]

def predict_units(X_train, y_train, X_predict):
    water_prefixes = ('weekly_inflow_', 'inflow_ratio_', 'headroom_')
    core = [c for c in X_train if not c.startswith(water_prefixes)]
    local_prefixes = ('volume_', 'inflow_', 'margin_', 'weekly_margin_', 'profitable_share_', 'local_profitable_')
    shared = [c for c in core if not c.startswith(local_prefixes)]
    def fit_unit(j):
        plant = units[j].rsplit('_G', 1)[0]
        names = reservoirs[plant]
        local = [c for c in core if c in shared or any(c.endswith('_' + name) for name in names)]
        columns = core if plant in ('Hogga', 'Lio', 'Kjela') else local
        settings = dict(n_estimators=40, max_depth=20, min_samples_leaf=5, max_samples=0.25, max_features=0.7)
        if plant == 'Haukeli':
            columns = list(X_train.columns)
            settings = dict(n_estimators=60, min_samples_leaf=2, max_samples=0.35)
        elif plant == 'Vinje':
            settings = dict(n_estimators=100, max_samples=0.5)
        model = RandomForestClassifier(random_state=67, **settings)
        model.fit(X_train[columns], y_train[:, j])
        probabilities = model.predict_proba(X_predict[columns])[:, 1]
        if plant == 'Lio':
            deeper = RandomForestClassifier(n_estimators=100, max_samples=0.5, random_state=67)
            deeper.fit(X_train[local], y_train[:, j])
            probabilities = (probabilities + deeper.predict_proba(X_predict[local])[:, 1]) / 2
        return probabilities
    with ThreadPoolExecutor(max_workers=3) as pool:
        return np.column_stack(list(pool.map(fit_unit, range(14))))

X = make_features(train)
y = train[target_columns].to_numpy(dtype=np.uint8).reshape(-1, 14, HORIZON)
y = y.transpose(0, 2, 1).reshape(-1, 14)
train_mask = train['starttime'] < pd.Timestamp('2022-01-01') - pd.Timedelta(days=7)
valid_mask = train['starttime'].between('2022-01-01', '2022-12-31')
train_rows = np.repeat(train_mask.to_numpy(), HORIZON)
valid_rows = np.repeat(valid_mask.to_numpy(), HORIZON)

probabilities = predict_units(X.loc[train_rows], y[train_rows], X.loc[valid_rows])
wide = probabilities.reshape(-1, HORIZON, 14).transpose(0, 2, 1).reshape(-1, len(target_columns))
validation = pd.DataFrame(wide, columns=target_columns)
validation.insert(0, 'Run No', train.loc[valid_mask, 'Run No'].to_numpy())
auc = score(train.loc[valid_mask, ['Run No', *target_columns]], validation)
print(f'2022 validation micro ROC-AUC: {auc:.4f}')


2022 validation micro ROC-AUC: 0.9794


## 4. Generate the submission

Refit on all labeled cases and save probabilities in the sample submission order.

In [ ]:
test = pd.read_csv('prediction_mapping.csv', parse_dates=['starttime'])
sample = pd.read_csv('sample_submission.csv')
test = test.set_index('Run No').loc[sample['Run No']].reset_index()
probabilities = predict_units(X, y, make_features(test))
wide = probabilities.reshape(-1, HORIZON, 14).transpose(0, 2, 1).reshape(-1, len(target_columns))
submission = pd.DataFrame(wide, columns=target_columns)
submission.insert(0, 'Run No', test['Run No'].to_numpy())

output = Path('submissions') / f'{pd.Timestamp.today():%Y-%m-%d}_model3_cv{auc:.4f}.csv'
output.parent.mkdir(exist_ok=True)
submission.loc[:, sample.columns].to_csv(output, index=False)
print(f'Saved {output}')


KeyboardInterrupt: 